In [1]:
import tarfile
from pathlib import Path

import numpy as np
import tensorflow as tf
FLOWER_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

print("TensorFlow 版本:", tf.__version__)
DATA_DIR = None
EXPORT_DIR = "exported_flower_model"
EPOCHS = 5
BATCH_SIZE = 32
IMAGE_SIZE = 224
LEARNING_RATE = 1e-3
QUANTIZATION = "dynamic"
SEED = 123


TensorFlow 版本: 2.15.0


In [2]:
def load_flower_datasets(data_dir, image_size, batch_size, seed):
    
    if data_dir is None:
        archive_path = tf.keras.utils.get_file(
            "flower_photos.tgz",
            FLOWER_URL,
            extract=False,
        )
        archive_path = Path(archive_path)


        candidates = [
            archive_path.parent / "flower_photos",
            archive_path.parent / "flower_photos_extracted" / "flower_photos",
        ]
        data_dir = next((path for path in candidates if path.exists()), None)
        if data_dir is None:
            with tarfile.open(archive_path, "r:gz") as tar:
                tar.extractall(archive_path.parent / "flower_photos_extracted")
            data_dir = archive_path.parent / "flower_photos_extracted" / "flower_photos"
    else:
        data_dir = Path(data_dir)


    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=seed,
        image_size=(image_size, image_size),
        batch_size=batch_size,
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=seed,
        image_size=(image_size, image_size),
        batch_size=batch_size,
    )
    class_names = train_ds.class_names

    val_batches = int(tf.data.experimental.cardinality(val_ds).numpy())
    test_ds = val_ds.take(val_batches // 2)
    val_ds = val_ds.skip(val_batches // 2)


    autotune = tf.data.AUTOTUNE
    train_ds = train_ds.cache().shuffle(1000, seed=seed).prefetch(autotune)
    val_ds = val_ds.cache().prefetch(autotune)
    test_ds = test_ds.cache().prefetch(autotune)
    return train_ds, val_ds, test_ds, class_names


train_ds, val_ds, test_ds, class_names = load_flower_datasets(
    DATA_DIR,
    IMAGE_SIZE,
    BATCH_SIZE,
    SEED,
)

print("类别数量:", len(class_names))
print("类别名称:", class_names)


Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
类别数量: 5
类别名称: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


In [3]:
def build_model(num_classes, image_size, learning_rate):
    
    inputs = tf.keras.Input(shape=(image_size, image_size, 3), name="image")

   
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

  
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(image_size, image_size, 3),
        include_top=False,
        weights="imagenet",
        pooling="avg",
    )

   
    base_model.trainable = False
    x = base_model(x, training=False)
    x = tf.keras.layers.Dropout(0.2)(x)

  
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
    )
    return model
    
model = build_model(len(class_names), IMAGE_SIZE, LEARNING_RATE)
model.summary()



Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 image (InputLayer)          [(None, 224, 224, 3)]     0         
                                                                 
 tf.math.truediv (TFOpLambd  (None, 224, 224, 3)       0         
 a)                                                              
                                                                 
 tf.math.subtract (TFOpLamb  (None, 224, 224, 3)       0         
 da)                                                             
                                                                 
 mobilenetv2_1.00_224 (Func  (None, 1280)              2257984   
 tional)                                                         
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                           

In [4]:
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

Epoch 1/5


92/92 [==============================] - 16s 149ms/step - loss: 0.8288 - accuracy: 0.6877 - val_loss: 0.4324 - val_accuracy: 0.8639
Epoch 2/5
92/92 [==============================] - 12s 136ms/step - loss: 0.4188 - accuracy: 0.8552 - val_loss: 0.3530 - val_accuracy: 0.8874
Epoch 3/5
92/92 [==============================] - 12s 133ms/step - loss: 0.3390 - accuracy: 0.8804 - val_loss: 0.3098 - val_accuracy: 0.8953
Epoch 4/5
92/92 [==============================] - 12s 133ms/step - loss: 0.2857 - accuracy: 0.9074 - val_loss: 0.2895 - val_accuracy: 0.9005
Epoch 5/5
92/92 [==============================] - 12s 134ms/step - loss: 0.2455 - accuracy: 0.9186 - val_loss: 0.2672 - val_accuracy: 0.9162


In [5]:
loss, accuracy = model.evaluate(test_ds)
print(f"test_loss={loss:.4f}, test_accuracy={accuracy:.4f}")

11/11 [==============================] - 1s 122ms/step - loss: 0.3165 - accuracy: 0.8977
test_loss=0.3165, test_accuracy=0.8977


In [6]:
def convert_to_tflite(model, quantization, representative_ds):
   
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    if quantization == "dynamic":
       
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    elif quantization == "float16":
      
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    elif quantization == "int8":
       
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        def representative_data_gen():
            for images, _ in representative_ds.take(100):
                for image in images:
                    yield [tf.expand_dims(tf.cast(image, tf.float32), 0)]

        converter.representative_dataset = representative_data_gen
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.uint8
        converter.inference_output_type = tf.uint8
    elif quantization != "none":
        raise ValueError(f"Unsupported quantization mode: {quantization}")

    return converter.convert()

   
export_dir = Path(EXPORT_DIR)
export_dir.mkdir(parents=True, exist_ok=True)


labels_path = export_dir / "labels.txt"
labels_path.write_text("\n".join(class_names) + "\n", encoding="utf-8")


keras_path = export_dir / "flower_classifier.keras"
model.save(keras_path)


tflite_model = convert_to_tflite(model, QUANTIZATION, train_ds)
tflite_path = export_dir / "model.tflite"
tflite_path.write_bytes(tflite_model)

print(f"已保存 Keras 模型: {keras_path}")
print(f"已保存 TFLite 模型: {tflite_path}")
print(f"已保存标签文件: {labels_path}")

    


INFO:tensorflow:Assets written to: C:\Users\26746\AppData\Local\Temp\tmpw8oyrj3s\assets


INFO:tensorflow:Assets written to: C:\Users\26746\AppData\Local\Temp\tmpw8oyrj3s\assets


已保存 Keras 模型: exported_flower_model\flower_classifier.keras
已保存 TFLite 模型: exported_flower_model\model.tflite
已保存标签文件: exported_flower_model\labels.txt


In [7]:
def smoke_test_tflite(tflite_path, test_ds, class_names):
    # 加载 TFLite 模型并分配张量内存。
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    images, labels = next(iter(test_ds.unbatch().batch(8)))
    input_data = tf.cast(images, input_details["dtype"]).numpy()

    # 如果模型是 uint8 输入，需要按照量化参数把图片转换到对应范围。
    if input_details["dtype"] == np.uint8:
        scale, zero_point = input_details["quantization"]
        if scale:
            input_data = images.numpy() / scale + zero_point
            input_data = np.clip(input_data, 0, 255).astype(np.uint8)

    predictions = []
    for image in input_data:
        interpreter.set_tensor(input_details["index"], np.expand_dims(image, 0))
        interpreter.invoke()
        predictions.append(interpreter.get_tensor(output_details["index"])[0])

    predicted_ids = np.argmax(np.asarray(predictions), axis=1)
    for expected, predicted in zip(labels.numpy()[:5], predicted_ids[:5]):
        print(f"真实类别={class_names[expected]}, 预测类别={class_names[predicted]}")


In [8]:
smoke_test_tflite(tflite_path, test_ds, class_names)

真实类别=sunflowers, 预测类别=sunflowers
真实类别=roses, 预测类别=daisy
真实类别=tulips, 预测类别=tulips
真实类别=sunflowers, 预测类别=sunflowers
真实类别=dandelion, 预测类别=dandelion
